# Day 5 — AI Harness and Automation

## Daily project: Mini AI Harness + Website Maintenance Agent

This is the classroom master notebook for Day 5. Work from top to bottom: each section introduces one limitation, adds one system layer, and carries that improvement into the daily project.

### How to use this notebook

- Run the environment check and setup cells before beginning.
- Complete sections in order during class; optional provider comparisons are clearly marked.
- If Colab restarts, rerun the current section's import/setup cell before continuing.
- At each checkpoint, explain the observable change before moving forward.
- Use mock or cached mode first. Use the instructor-issued OpenRouter credit only for bounded live observations.

### Day 5 contents

1. [1. What is an AI harness?](#day-5-section-1)
2. [2. Model configuration and runtime](#day-5-section-2)
3. [3. Tool registry and discovery](#day-5-section-3)
4. [4. Permissions, approval, and limits](#day-5-section-4)
5. [5. Events, logs, and checkpoints](#day-5-section-5)
6. [6. MCP client: discover, then govern](#day-5-section-6)
7. [7. Capstone: Mini AI Harness](#day-5-section-7)
8. [Build a Capability-Aware Tool Registry](#day-5-section-8)
9. [9. Operational capstone: Website Maintenance Agent](#day-5-section-9)

---


<a id="day-5-section-1"></a>

## 5.1 — 1. What is an AI harness?

Across Days 1–4 we repeatedly configured a model, described tools, ran a loop, applied policy, stored state, and logged events. A **runtime** executes one run. A **harness** packages these reusable responsibilities so multiple agent configurations can run consistently.

An agent is the configured behavior. A framework is a coding library. A protocol is an interoperability contract. None of these words guarantees safety or quality.


## Before you begin

**Required — all students:** run mock mode first. **Choose one:** repeat provider lessons with OpenRouter when configured. The real MCP stdio cell requires the pinned Day 5 SDK; fake MCP is the fallback.

### Learning outcomes

Identify repeated responsibilities across earlier projects and distinguish agent, workflow, framework, protocol, runtime, and harness.

Architecture reference: [Day 5 diagrams D16](../../diagrams/source/day_05.md).

### Expected observation

Two configurations differ while pointing to the same runtime responsibilities. Exact IDs, timing, and live wording will vary.

## Concept briefing

## Why consolidate the earlier projects

By Day 5, several applications repeat the same responsibilities: load model
configuration, describe tools, validate arguments, enforce policy, limit steps, record
events and save continuation state. Copying this code into every agent makes safety fixes
inconsistent. A reusable runtime centralises the execution lifecycle.

This course uses **harness** as an umbrella term for the environment around an agent. In
industry, related terms include agent runtime, orchestration layer and agent platform.
The exact vocabulary varies; the responsibilities are transferable.


In [ ]:
from pathlib import Path
import sys,json
DAY=Path.cwd()
if (DAY/"day_05_ai_harness").exists(): DAY=DAY/"day_05_ai_harness"
elif DAY.name=="notebooks": DAY=DAY.parent
if not (DAY/"src"/"mini_harness").exists(): raise RuntimeError("Launch Jupyter from the repository, day folder, or notebooks folder.")
sys.path.insert(0,str(DAY/"src"))
from mini_harness import *
def load_config(name):
    raw=json.loads((DAY/"configs"/f"{name}.json").read_text(encoding="utf-8"))
    raw["model"]=ModelConfig(**raw["model"])
    return AgentConfig(**raw)
print("Day folder:",DAY)

In [ ]:
research=load_config("research_agent"); task=load_config("task_agent")
print(research)
print(task)
print("Different behavior; same runtime contract.")

## Architecture inventory

Map each recurring concern to one module: configuration, provider, registry, policy, runtime, events, checkpoints, memory, and MCP adapter. We build the smallest useful harness, not a general-purpose coding platform.

## Your turn

List three duplicated responsibilities from Days 1 and 3 and decide which belongs in shared infrastructure.

## Recap

A harness standardizes execution concerns without erasing application policy. Name one responsibility that deliberately remains application-specific.

---

### Section 5.1 checkpoint

Before continuing, confirm that you can state: **what limitation we observed, what layer we added, and what evidence shows the improvement worked.**


<a id="day-5-section-2"></a>

## 5.2 — 2. Model configuration and runtime

Configuration is data; execution is code. Centralizing provider selection prevents every agent from reinventing API calls and makes mock/OpenRouter/Ollama switching explicit.


## Before you begin

**Required — all students:** run mock mode first. **Choose one:** repeat provider lessons with OpenRouter when configured. The real MCP stdio cell requires the pinned Day 5 SDK; fake MCP is the fallback.

### Learning outcomes

Load provider configuration, run the same runtime in mock or OpenRouter mode, and observe provider failures as events.

Architecture reference: [Day 5 diagrams D16](../../diagrams/source/day_05.md).

### Expected observation

Mock mode completes locally; configured OpenRouter uses the same runtime contract and records token usage. Exact IDs, timing, and live wording will vary.

## Concept briefing

## Configuration versus runtime

An agent configuration describes application-specific behavior: instructions, allowed
tools, model settings and limits. The runtime executes that configuration. A research
agent and a safe task agent should use one runtime without sharing inappropriate tools or
permissions.

A provider adapter hides API-specific request and response shapes behind a small
interface. Switching mock, OpenRouter or Ollama should not rewrite policy or the registry.
Provider metadata such as tokens, cost, latency and errors should still be preserved in
events.


In [ ]:
from pathlib import Path
import sys,json
DAY=Path.cwd()
if (DAY/"day_05_ai_harness").exists(): DAY=DAY/"day_05_ai_harness"
elif DAY.name=="notebooks": DAY=DAY.parent
if not (DAY/"src"/"mini_harness").exists(): raise RuntimeError("Launch Jupyter from the repository, day folder, or notebooks folder.")
sys.path.insert(0,str(DAY/"src"))
from mini_harness import *
def load_config(name):
    raw=json.loads((DAY/"configs"/f"{name}.json").read_text(encoding="utf-8"))
    raw["model"]=ModelConfig(**raw["model"])
    return AgentConfig(**raw)
print("Day folder:",DAY)

In [ ]:
import os
registry=build_demo_registry(); research=load_config("research_agent")
if os.getenv("OPENROUTER_API_KEY"):
    research.model.provider="openrouter"; research.model.model=os.getenv("OPENROUTER_MODEL","openai/gpt-oss-120b")
provider=build_provider(research.model); print("Provider:",research.model.provider)
runtime=HarnessRuntime(registry,provider)
result=runtime.run(research,"What is centralized by a harness?")
print(result.status,result.output)
for event in result.events: print(event)

## Live provider exercise

`build_provider` now supplies the tested OpenAI-compatible adapter. Change only `ModelConfig.provider`; registry, policy, and runtime remain unchanged. Ollama remains optional. Mock mode tests orchestration—it does not assess answer quality.

### Boundaries

Temperature and output limits belong in model configuration. Maximum tool steps belongs in agent/runtime configuration. API keys belong in environment variables, never JSON or notebooks.

## Your turn

Switch only the provider configuration, then compare event shapes rather than answer wording.

## Recap

Provider adapters isolate API differences from agent behavior. Name one responsibility that deliberately remains application-specific.

---

### Section 5.2 checkpoint

Before continuing, confirm that you can state: **what limitation we observed, what layer we added, and what evidence shows the improvement worked.**


<a id="day-5-section-3"></a>

## 5.3 — 3. Tool registry and discovery

A registry separates capability definition from the loop. Each tool carries a name, description, input schema, and risk level. The agent sees only its allow-listed subset.


## Before you begin

**Required — all students:** run mock mode first. **Choose one:** repeat provider lessons with OpenRouter when configured. The real MCP stdio cell requires the pinned Day 5 SDK; fake MCP is the fallback.

### Learning outcomes

Register tools dynamically, discover an allow-listed subset, and validate arguments before execution.

Architecture reference: [Day 5 diagrams D16](../../diagrams/source/day_05.md).

### Expected observation

Research sees only lookup_notes; a missing draft body is rejected before the function runs. Exact IDs, timing, and live wording will vary.

## Concept briefing

## Registry, validation and policy

A tool registry stores names, descriptions, input schemas, executors and local risk
classifications. Discovery answers "what capabilities are visible?" Validation answers
"are these arguments structurally acceptable?" Policy answers "may this agent execute
this action now?" These are separate decisions.

The runtime should fail closed on unknown tools, invalid arguments and disallowed actions.
It should never ask the same model that proposed an action to make the authoritative
permission decision.


In [ ]:
from pathlib import Path
import sys,json
DAY=Path.cwd()
if (DAY/"day_05_ai_harness").exists(): DAY=DAY/"day_05_ai_harness"
elif DAY.name=="notebooks": DAY=DAY.parent
if not (DAY/"src"/"mini_harness").exists(): raise RuntimeError("Launch Jupyter from the repository, day folder, or notebooks folder.")
sys.path.insert(0,str(DAY/"src"))
from mini_harness import *
def load_config(name):
    raw=json.loads((DAY/"configs"/f"{name}.json").read_text(encoding="utf-8"))
    raw["model"]=ModelConfig(**raw["model"])
    return AgentConfig(**raw)
print("Day folder:",DAY)

In [ ]:
registry=build_demo_registry()
for spec in registry.discover(): print(spec)
research=load_config("research_agent")
print("Research sees:",[x.name for x in registry.discover(research.allowed_tools)])

In [ ]:
print(registry.call("lookup_notes",{"query":"harness"}))
try: registry.call("create_draft",{"subject":"body is missing"})
except Exception as error: print(type(error).__name__,error)

## Important distinction

JSON Schema describes valid arguments; it does not authorize execution. Tool discovery says a capability exists; it does not say this agent or user may call it. Policy is checked after validation and before side effects.

## Your turn

Register one read-only tool with a required string argument and prove an invalid type fails.

## Recap

Schemas describe valid calls; registry discovery does not grant authority. Name one responsibility that deliberately remains application-specific.

---

### Section 5.3 checkpoint

Before continuing, confirm that you can state: **what limitation we observed, what layer we added, and what evidence shows the improvement worked.**


<a id="day-5-section-4"></a>

## 5.4 — 4. Permissions, approval, and limits

The model proposes; Python disposes. Read and reversible local writes may run, external effects pause, destructive tools are denied, and unknown tools fail closed. Every loop has a hard step maximum.


## Before you begin

**Required — all students:** run mock mode first. **Choose one:** repeat provider lessons with OpenRouter when configured. The real MCP stdio cell requires the pinned Day 5 SDK; fake MCP is the fallback.

### Learning outcomes

Enforce risk policy, pause external actions, cancel safely, deny destructive actions, and stop loops.

Architecture reference: [Day 5 diagrams D16](../../diagrams/source/day_05.md).

### Expected observation

Send pauses, rejection becomes cancelled, destructive access is denied, and an endless loop reaches step_limit. Exact IDs, timing, and live wording will vary.

In [ ]:
from pathlib import Path
import sys,json
DAY=Path.cwd()
if (DAY/"day_05_ai_harness").exists(): DAY=DAY/"day_05_ai_harness"
elif DAY.name=="notebooks": DAY=DAY.parent
if not (DAY/"src"/"mini_harness").exists(): raise RuntimeError("Launch Jupyter from the repository, day folder, or notebooks folder.")
sys.path.insert(0,str(DAY/"src"))
from mini_harness import *
def load_config(name):
    raw=json.loads((DAY/"configs"/f"{name}.json").read_text(encoding="utf-8"))
    raw["model"]=ModelConfig(**raw["model"])
    return AgentConfig(**raw)
print("Day folder:",DAY)

In [ ]:
runtime=HarnessRuntime(build_demo_registry(),MockModel()); task=load_config("task_agent")
pending=runtime.run(task,"Send a synthetic course update")
print(pending.status,pending.pending_action)
print("Checkpoint:",runtime.checkpoints.load(pending.run_id))
rejected=runtime.resume(pending.run_id,task,approved=False)
print(rejected.status,rejected.output)

In [ ]:
class EndlessModel:
    def decide(self,prompt,config,tools,history):
        return ModelDecision("tool",tool="lookup_notes",arguments={"query":prompt})
cfg=load_config("research_agent"); cfg.max_steps=2
limited=HarnessRuntime(build_demo_registry(),EndlessModel()).run(cfg,"keep going")
print(limited.status,limited.events[-1])

In [ ]:
cfg=load_config("task_agent"); cfg.allowed_tools.append("erase_workspace")
destructive=build_demo_registry().get("erase_workspace").spec
print("Visible in allow-list, but policy decision is:",decide(cfg,destructive))
assert decide(cfg,destructive)=="deny"

Rejection is a successful safety outcome even though the run status is failed in this minimal implementation. A production schema might distinguish `cancelled` from technical failure.

## Your turn

Add erase_workspace to a temporary agent allow-list and prove risk policy still denies it.

## Recap

The model proposes; policy and limits control execution. Name one responsibility that deliberately remains application-specific.

---

### Section 5.4 checkpoint

Before continuing, confirm that you can state: **what limitation we observed, what layer we added, and what evidence shows the improvement worked.**


<a id="day-5-section-5"></a>

## 5.5 — 5. Events, logs, and checkpoints

Events are append-only observations; checkpoints are mutable continuation state. Events support audit and evaluation. A checkpoint lets approval resume the exact pending action without asking the model to recreate it.


## Before you begin

**Required — all students:** run mock mode first. **Choose one:** repeat provider lessons with OpenRouter when configured. The real MCP stdio cell requires the pinned Day 5 SDK; fake MCP is the fallback.

### Learning outcomes

Distinguish events from checkpoints and prove a durable checkpoint can resume after runtime reconstruction.

Architecture reference: [Day 5 diagrams D16](../../diagrams/source/day_05.md).

### Expected observation

Approval state survives a new JSON checkpoint-store instance and clears after resolution. Exact IDs, timing, and live wording will vary.

## Concept briefing

## Events and checkpoints

Events are append-only observations such as run started, model completed, policy decided
and tool completed. A trace groups events belonging to one run. A checkpoint stores
continuation state so a paused run can resume.

A checkpoint is not an audit log, and an event log is not enough to resume execution.
Durable approval needs the exact pending action and a stable run identifier. Sensitive
arguments should be redacted or omitted from telemetry where possible.

## Retries, timeouts and retry budgets

Network calls fail. A runtime should apply a timeout and may retry transient failures such
as temporary rate limits. Retries must be bounded and recorded. Exponential backoff with
jitter helps avoid many clients retrying simultaneously.

Do not retry every failure. Invalid arguments, policy denial and most authentication
errors will not improve on repetition. Consequential tools require an idempotency strategy
before automatic retry. The runtime should have both a step budget and a retry budget so
one failing provider does not consume unlimited time or credit.

## Cost attribution

Record model, prompt/configuration version, input tokens, output tokens, reasoning tokens,
estimated cost and run ID. This makes it possible to compare agents and enforce classroom
budgets. Cost belongs to the complete run, including retries and specialist calls, not
only the final response.

The included API credit is a controlled learning resource. Mock mode should be used while
debugging application logic; live calls should be used when model behavior is the subject
of the exercise.


In [ ]:
from pathlib import Path
import sys,json
DAY=Path.cwd()
if (DAY/"day_05_ai_harness").exists(): DAY=DAY/"day_05_ai_harness"
elif DAY.name=="notebooks": DAY=DAY.parent
if not (DAY/"src"/"mini_harness").exists(): raise RuntimeError("Launch Jupyter from the repository, day folder, or notebooks folder.")
sys.path.insert(0,str(DAY/"src"))
from mini_harness import *
def load_config(name):
    raw=json.loads((DAY/"configs"/f"{name}.json").read_text(encoding="utf-8"))
    raw["model"]=ModelConfig(**raw["model"])
    return AgentConfig(**raw)
print("Day folder:",DAY)

In [ ]:
events=EventStore(); checkpoint_dir=DAY/"data"/"demo_checkpoints"
checkpoints=JSONCheckpointStore(checkpoint_dir)
runtime=HarnessRuntime(build_demo_registry(),MockModel(),events,checkpoints)
cfg=load_config("task_agent"); paused=runtime.run(cfg,"Send the synthetic update")
for event in events.get(paused.run_id): print(event)
print("Saved state:",checkpoints.load(paused.run_id))
# Simulate a restart by constructing a new runtime and checkpoint-store object.
resumed_runtime=HarnessRuntime(build_demo_registry(),MockModel(),events,JSONCheckpointStore(checkpoint_dir))
done=resumed_runtime.resume(paused.run_id,cfg,approved=True)
print("Final:",done.status,done.output)
print("Checkpoint cleared:",checkpoints.load(paused.run_id))

## Persisting locally

Pass a JSONL path to `EventStore` for durable logs. The teaching checkpoint store is in-memory and transparent; replacing it with SQLite is a useful extension. Do not log secrets, private prompts, or hidden reasoning. LangSmith export remains optional and synthetic-only.

## Your turn

Inspect the checkpoint file, reconstruct the runtime, resume, and verify the event order.

## Recap

Events explain history; checkpoints preserve continuation state. Name one responsibility that deliberately remains application-specific.

---

### Section 5.5 checkpoint

Before continuing, confirm that you can state: **what limitation we observed, what layer we added, and what evidence shows the improvement worked.**


<a id="day-5-section-6"></a>

## 5.6 — 6. MCP client: discover, then govern

Model Context Protocol standardizes how clients discover and invoke server capabilities. It does not decide whether a capability is trusted or authorized. We first use an offline protocol-shaped client, then optionally connect to the instructor’s local stdio server.


## Before you begin

**Required — all students:** run mock mode first. **Choose one:** repeat provider lessons with OpenRouter when configured. The real MCP stdio cell requires the pinned Day 5 SDK; fake MCP is the fallback.

### Learning outcomes

Discover an MCP tool, classify it locally, authorize it through harness policy, and invoke it through mock or stdio transport.

Architecture reference: [Day 5 diagrams D17](../../diagrams/source/day_05.md).

### Expected observation

course_lookup is discovered, classified read-only, and called only after local validation and policy. Exact IDs, timing, and live wording will vary.

## Concept briefing

## MCP: protocol, not permission

Model Context Protocol lets a client initialise a session, discover server capabilities
and invoke them through a common contract. A server may expose tools, resources or prompts.
The protocol improves interoperability; it does not establish trust.

An MCP tool description and its results are untrusted external content. Before importing
a discovered tool, the harness should consider server origin, schema, local risk,
permitted agents, arguments, timeout, output handling and logging. A server changing its
advertised tools must not silently expand application authority.

The Day 5 rule is therefore:

```text
discovery is not authorization
```

The client discovers the tool, the harness classifies it, local policy authorises or
pauses it, and only then does the protocol call occur.


In [ ]:
from pathlib import Path
import sys,json
DAY=Path.cwd()
if (DAY/"day_05_ai_harness").exists(): DAY=DAY/"day_05_ai_harness"
elif DAY.name=="notebooks": DAY=DAY.parent
if not (DAY/"src"/"mini_harness").exists(): raise RuntimeError("Launch Jupyter from the repository, day folder, or notebooks folder.")
sys.path.insert(0,str(DAY/"src"))
from mini_harness import *
def load_config(name):
    raw=json.loads((DAY/"configs"/f"{name}.json").read_text(encoding="utf-8"))
    raw["model"]=ModelConfig(**raw["model"])
    return AgentConfig(**raw)
print("Day folder:",DAY)

In [ ]:
import asyncio
async def offline_demo():
    client=FakeMCPClient(); tools=await client.list_tools()
    print("Discovered:",tools)
    raw=tools[0]
    spec=ToolSpec(raw["name"],raw["description"],raw["inputSchema"],"read")
    cfg=AgentConfig("mcp_demo","Use one supplied fact.",[spec.name])
    print("Local policy:",decide(cfg,spec))
    if decide(cfg,spec)=="allow": print("Result:",await client.call_tool(spec.name,{"topic":"mcp"}))
await offline_demo()

## Official SDK local lab

Install the instructor-pinned stable SDK (`mcp[cli]`). The supplied server is course infrastructure; students need not write it. `StdioMCPClient` uses `StdioServerParameters`, `stdio_client`, `ClientSession.initialize()`, `list_tools()`, and `call_tool()`. On Windows use the Python executable that launches the current environment.

In [ ]:
# Set RUN_REAL_MCP=1 before launching Jupyter after installing the pinned SDK.
import importlib.util,os,sys
if os.getenv("RUN_REAL_MCP")=="1" and importlib.util.find_spec("mcp"):
    client=StdioMCPClient(sys.executable,[str(DAY/"instructor_mcp_server.py")])
    tools,result=await client.list_and_optionally_call("course_lookup",{"topic":"harness"})
    print([tool.name for tool in tools]); print(result)
else:
    print("Real MCP skipped. Complete the offline policy path above or enable RUN_REAL_MCP=1.")

## Security checkpoint

Before importing an MCP tool into the registry, inspect server origin, tool description/schema, local risk classification, allowed agents, arguments, output handling, timeout, and logging. A server can change its advertised tools; rediscovery is not automatic authorization.

## Your turn

Change its local risk to external and show that discovery stays identical while authorization changes.

## Recap

MCP standardizes capability exchange; the harness retains trust and permission decisions. Name one responsibility that deliberately remains application-specific.

---

### Section 5.6 checkpoint

Before continuing, confirm that you can state: **what limitation we observed, what layer we added, and what evidence shows the improvement worked.**


<a id="day-5-section-7"></a>

## 5.7 — 7. Capstone: Mini AI Harness

One runtime now hosts a research agent and a safe task agent. Demonstrate configuration loading, scoped discovery, validation, policy, bounded execution, events, approval/checkpoint resume, memory interface, and MCP discovery.


## Before you begin

**Required — all students:** run mock mode first. **Choose one:** repeat provider lessons with OpenRouter when configured. The real MCP stdio cell requires the pinned Day 5 SDK; fake MCP is the fallback.

### Learning outcomes

Host two agent configurations with one live/mock runtime, registry, policy, memory, events, checkpoints, and MCP boundary.

Architecture reference: [Day 5 diagrams D16–D18](../../diagrams/source/day_05.md).

### Expected observation

Research completes with evidence; task sending pauses; the explicit resume decision determines the outcome. Exact IDs, timing, and live wording will vary.

## Concept briefing

## Mapping the course to production systems

| Course term | Common production terminology |
|---|---|
| Provider adapter | model client/provider layer |
| Agent configuration | agent definition/profile |
| Harness runtime | agent runtime/orchestration layer |
| Tool registry | tool/plugin registry |
| Policy | authorization or guardrail middleware |
| Events | tracing/telemetry |
| Checkpoint store | durable execution/state persistence |
| MCP client | protocol integration layer |

Production SDKs package different subsets of these responsibilities. Students should be
able to open an unfamiliar SDK and locate where its model calls, tools, policy, state and
events live rather than assuming the SDK itself is the architecture.

## What the mini harness does not provide

The classroom harness is intentionally not a production platform. It does not provide
enterprise identity, operating-system sandboxing, remote MCP authentication, distributed
workers, deployment or guaranteed model quality. Its purpose is to make the essential
boundaries visible so students can recognise and evaluate larger systems later.


In [ ]:
from pathlib import Path
import sys,json
DAY=Path.cwd()
if (DAY/"day_05_ai_harness").exists(): DAY=DAY/"day_05_ai_harness"
elif DAY.name=="notebooks": DAY=DAY.parent
if not (DAY/"src"/"mini_harness").exists(): raise RuntimeError("Launch Jupyter from the repository, day folder, or notebooks folder.")
sys.path.insert(0,str(DAY/"src"))
from mini_harness import *
def load_config(name):
    raw=json.loads((DAY/"configs"/f"{name}.json").read_text(encoding="utf-8"))
    raw["model"]=ModelConfig(**raw["model"])
    return AgentConfig(**raw)
print("Day folder:",DAY)

In [ ]:
runtime=HarnessRuntime(build_demo_registry(),MockModel())
cases=[("research_agent","What is a harness?"),("task_agent","Prepare a concise project update")]
for name,prompt in cases:
    result=runtime.run(load_config(name),prompt)
    print(name,result.status,result.output)
    print("events:",[e["event"] for e in result.events])

In [ ]:
task=load_config("task_agent")
pending=runtime.run(task,"Send a synthetic project update")
assert pending.status=="pending_approval"
print("Approval card:",pending.pending_action)
final=runtime.resume(pending.run_id,task,approved=True)
print(final.status,final.output)
print([e["event"] for e in final.events])

In [ ]:
memory=SimpleMemory(); memory.add("fictional_asha","Prefer concise project updates")
print(memory.search("fictional_asha","concise update"))
async def mcp_check():
    client=FakeMCPClient(); return await client.list_tools(),await client.call_tool("course_lookup",{"topic":"harness"})
tools,mcp_result=await mcp_check(); print(tools,mcp_result)

## Final explanation

Draw: **agent config → runtime → provider/tool proposal → registry validation → policy → approval or execution → events/checkpoint**. MCP enters through discovery/invocation but still passes local policy.

Defend what this harness does *not* provide: authentication, OS sandboxing, remote MCP trust, distributed workers, deployment, or guaranteed model quality. Optional next steps are FastAPI, Docker, SQLite checkpoints, LangSmith/OpenTelemetry export, and a second hosted provider—not core requirements.

## Your turn

Add a third configuration without modifying runtime.py and submit its event trace plus one denied action.

## Recap

The harness is reusable infrastructure, not a universal autonomous agent. Name one responsibility that deliberately remains application-specific.

---

### Section 5.7 checkpoint

Before continuing, confirm that you can state: **what limitation we observed, what layer we added, and what evidence shows the improvement worked.**


<a id="day-5-section-8"></a>

## 5.8 — Build a Capability-Aware Tool Registry

This is an individual implementation lab. It uses no API key.


## Why this mechanism matters

A harness needs one controlled place for tool discovery and dispatch. The registry connects model-visible schemas to host-owned handlers while policy limits which capabilities a configuration receives.

## Contract

Reject duplicate registrations. Show schemas only for granted capabilities. Reject unknown or ungranted dispatches before calling the handler.

Before coding, write one sentence predicting the easiest failure to make.

In [ ]:
class ToolRegistry:
    def __init__(self):
        self._tools = {}

    def register(self, name, schema, handler, capability):
        raise NotImplementedError("Complete registration")

    def schemas_for(self, granted_capabilities):
        raise NotImplementedError("Complete filtered discovery")

    def dispatch(self, name, arguments, granted_capabilities):
        raise NotImplementedError("Complete protected dispatch")

## Behavioural check

Run this only after completing the starter cell. A passing check proves the listed contract examples, not every possible input.

In [ ]:
registry = ToolRegistry()
registry.register("add", {"name": "add", "parameters": {"a": "number", "b": "number"}},
                  lambda a, b: a + b, "math.read")
assert len(registry.schemas_for({"math.read"})) == 1
assert registry.schemas_for(set()) == []
assert registry.dispatch("add", {"a": 4, "b": 5}, {"math.read"}) == 9
try:
    registry.dispatch("add", {"a": 1, "b": 1}, set())
except PermissionError:
    pass
else:
    raise AssertionError("An ungranted tool must not run")
print("PASS")

## Explain and extend

Why must filtered schemas and protected dispatch both exist? Add tests for duplicate registration and an unknown tool name.

---

### Section 5.8 checkpoint

Before continuing, confirm that you can state: **what limitation we observed, what layer we added, and what evidence shows the improvement worked.**


<a id="day-5-section-9"></a>

## 5.9 — 9. Operational capstone: Website Maintenance Agent

One bounded scheduled tick fetches an update, compares durable state, obtains a structured proposal, applies named guardrails, pauses for approval, writes a real local website file, verifies it, and records the run. The classroom target is local Markdown; direct public publishing is outside the core.


## Before you begin

**Required — all students:** run mock mode first. **Choose one:** repeat provider lessons with OpenRouter when configured. The real MCP stdio cell requires the pinned Day 5 SDK; fake MCP is the fallback.

### Learning outcomes

Run an operational cycle from public/cached update through state, proposal, named guardrails, approval, persistent website change, verification, and events.

Architecture reference: [Day 5 diagrams D19](../../diagrams/source/day_05.md).

### Expected observation

The clean update pauses before writing; rejection changes no file; approval creates a verified Markdown update; poisoned external instructions are blocked. Exact IDs, timing, and live wording will vary.

## Concept briefing

## Automation is a trigger, not intelligence

A scheduler can start a run every day, but scheduling alone is ordinary automation. The
agentic decision is whether new evidence warrants a change and which permitted action to
propose. Policy then decides whether the exact proposal may proceed.

The Website Maintenance Agent demonstrates a production-shaped cycle at classroom scale:
fetch a real or cached public update, compare it with durable processed-item state, create
a structured website proposal, apply guardrails, pause for approval, write a real local
file, verify the result and record events. The scheduler should call one bounded `check`
operation; it should not contain hidden business logic.

An optional LLM judge may score whether the proposed update is faithful to its source.
That judge belongs after deterministic checks and before approval or publication. It is
advisory because it can be inconsistent, biased toward fluent text or influenced by the
content it evaluates. File-path, schema, source, build and permission checks remain
authoritative application code.


In [ ]:
from pathlib import Path
import sys,json
DAY=Path.cwd()
if (DAY/"day_05_ai_harness").exists(): DAY=DAY/"day_05_ai_harness"
elif DAY.name=="notebooks": DAY=DAY.parent
if not (DAY/"src"/"mini_harness").exists(): raise RuntimeError("Launch Jupyter from the repository, day folder, or notebooks folder.")
sys.path.insert(0,str(DAY/"src"))
from mini_harness import *
def load_config(name):
    raw=json.loads((DAY/"configs"/f"{name}.json").read_text(encoding="utf-8"))
    raw["model"]=ModelConfig(**raw["model"])
    return AgentConfig(**raw)
print("Day folder:",DAY)

## 1. Configure a fresh classroom run

The cached source is repeatable. The optional live source reads public GitHub release data. Both produce the same `UpdateItem` contract.

In [ ]:
from mini_harness import (CachedJSONSource,GitHubReleaseSource,JSONStateStore,WebsiteGuardrails,
    WebsiteMaintenanceAgent,deterministic_proposer,OpenRouterWebsiteProposer,EventStore)
run_root=DAY/"data"/"website_classroom_run"
site_root=run_root/"site"
source=CachedJSONSource(DAY/"data"/"website_updates.json")
events=EventStore(run_root/"events.jsonl")
state=JSONStateStore(run_root/"state.json")
guardrails=WebsiteGuardrails(site_root,{"github.com"})
proposer=deterministic_proposer
print("Website target:",site_root)

## 2. Check once and inspect the exact proposal

No website file exists yet. Approval is a state transition over the exact saved proposal, not a conversational "yes".

In [ ]:
agent=WebsiteMaintenanceAgent(source,proposer,guardrails,state,events)
pending=agent.check_once()
print(pending.status,pending.proposal)
assert pending.status in {"pending_approval","no_change"}
print("Website exists before approval:",(site_root/"content"/"updates.md").exists())

## 3. Resolve deliberately

For the first run, leave `approved=False` and prove rejection has no side effect. Use a fresh run directory before repeating with approval.

In [ ]:
if pending.status=="pending_approval":
    approved=False  # change only after inspecting the proposal
    final=agent.resolve(pending.run_id,approved)
    print(final.status,final.message)
print("Website exists:",(site_root/"content"/"updates.md").exists())

## 4. Practical indirect prompt-injection challenge

The poisoned fixture mixes a real-looking update with instructions to reveal a key and invoke another tool. External content is evidence, not authority.

In [ ]:
poisoned=WebsiteMaintenanceAgent(
    CachedJSONSource(DAY/"data"/"poisoned_website_updates.json"),deterministic_proposer,
    guardrails,JSONStateStore(run_root/"poisoned_state.json"),events)
blocked=poisoned.check_once()
print(blocked.status,blocked.message)
assert blocked.status=="blocked"
assert not (site_root/"content"/"updates.md").exists()

## 5. Optional bounded live observations

Choose one live source and one live model call. If unavailable, use the cached source and instructor-captured trace.

```python
source = GitHubReleaseSource("modelcontextprotocol", "python-sdk", limit=3)
# proposer = OpenRouterWebsiteProposer()
```

Never publish automatically in this course. A live run stops at `pending_approval`.

## 6. Evaluation and optional LLM judge

Deterministic checks remain authoritative: trusted host, matching evidence, allowed path, body-size limit, prohibited active content, explicit approval and post-write verification. An optional LLM judge may score semantic faithfulness, but it is advisory and requires calibration against human-labelled examples.

A model council is unnecessary unless measured evidence shows one proposer/reviewer is inadequate. Day 4 provides the specialist-and-supervisor pattern.

## 7. Daily automation boundary

An operating-system scheduler, cron or CI schedule invokes `run_website_agent.py` once per day. Scheduling is ordinary automation; it merely triggers one bounded check. Production credentials, deployment and unattended approval are outside the core.

## Required live observation

Choose one bounded live observation: fetch up to three public releases or obtain one OpenRouter update proposal. Stop before approval. The cached source and captured trace are the outage fallback.


## Your turn

Run the cached cycle, reject once, approve once in a fresh state directory, and explain which controls remain authoritative with a live model.

## Recap

A scheduler triggers a bounded run; the agent proposes; guardrails and a human control the real side effect. Name one responsibility that deliberately remains application-specific.

---

### Section 5.9 checkpoint

Before continuing, confirm that you can state: **what limitation we observed, what layer we added, and what evidence shows the improvement worked.**


## Day 5 completion checklist

- [ ] I can explain how every section contributes to the **Mini AI Harness + Website Maintenance Agent**.
- [ ] I ran the deterministic path and at least one required live observation or classroom fallback.
- [ ] I completed the pivotal exercise without copying the reference implementation.
- [ ] I can identify the system state, safety boundary, and evidence used to judge the result.
